# 01. Daikibo Telemetry Data Analysis & Failure Clustering
**Objective:** Ingest semi-structured IoT sensor telemetry, compute unscheduled operational downtime, and pinpoint the primary root causes behind global equipment outages across Daikibo's manufacturing plants.


In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Plot styling
sns.set_theme(style="whitegrid")


## 1. Ingestion & Nested JSON Normalization
We load `daikibo-telemetry-data.json` from `../data/raw/` and flatten the nested `location` and `data` schema structures into a tabular DataFrame.


In [ ]:
raw_path = '../data/raw/daikibo-telemetry-data.json'

with open(raw_path, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

print(f"Total raw JSON records ingested: {len(raw_data):,}")

# Flatten nested JSON hierarchy
df_tel = pd.json_normalize(raw_data)
df_tel.head()


## 2. Feature Engineering: Downtime Mapping
Telemetry messages are broadcast at fixed 10-minute intervals. Each `'unhealthy'` status ping corresponds to 10 minutes of lost production time since the previous message. We construct a numerical measure `downtime_minutes`.


In [ ]:
# Construct downtime measure
df_tel['downtime_minutes'] = df_tel['data.status'].apply(lambda x: 10 if str(x).lower() == 'unhealthy' else 0)

# Convert epoch millisecond timestamp to standard datetime
df_tel['datetime'] = pd.to_datetime(df_tel['timestamp'], unit='ms')

# Inspect distribution
status_counts = df_tel['data.status'].value_counts()
print("Machine Status Counts:")
print(status_counts)
total_dt = df_tel['downtime_minutes'].sum()
print(f"\nTotal Enterprise Downtime: {total_dt:,} minutes ({total_dt/60:.2f} hours)")


## 3. Plant-Level Downtime Aggregations
Analyze which production hub is responsible for the largest share of unscheduled downtime.


In [ ]:
factory_summary = df_tel.groupby('location.factory').agg(
    total_records=('deviceID', 'count'),
    unhealthy_events=('downtime_minutes', lambda x: (x > 0).sum()),
    total_downtime_min=('downtime_minutes', 'sum')
).reset_index()

factory_summary['pct_of_global_downtime'] = (factory_summary['total_downtime_min'] / factory_summary['total_downtime_min'].sum()) * 100
factory_summary = factory_summary.sort_values(by='total_downtime_min', ascending=False)
factory_summary


## 4. Equipment Reliability Breakdown
Assess downtime distributions across device types to test whether failures are broadly distributed or concentrated in specific machinery.


In [ ]:
device_summary = df_tel.groupby('deviceType').agg(
    total_records=('deviceID', 'count'),
    unhealthy_events=('downtime_minutes', lambda x: (x > 0).sum()),
    total_downtime_min=('downtime_minutes', 'sum')
).reset_index()

device_summary['pct_of_global_downtime'] = (device_summary['total_downtime_min'] / device_summary['total_downtime_min'].sum()) * 100
device_summary = device_summary.sort_values(by='total_downtime_min', ascending=False)
device_summary


## 5. Cross-Tabulation Matrix (Facility x Device)
Identify the exact intersection of factories and failing equipment.


In [ ]:
pivot_downtime = df_tel.pivot_table(
    index='location.factory',
    columns='deviceType',
    values='downtime_minutes',
    aggfunc='sum',
    fill_value=0
)
pivot_downtime


## 6. Root Cause Hypothesis Testing: Temperature vs Status
Hypothesis: Failures were caused by thermal runaway or ambient overheating.


In [ ]:
# Check statistical distribution of temperature
print(df_tel.groupby('data.status')['data.temperature'].describe())

# Visual boxplot validation
plt.figure(figsize=(7, 4))
sns.boxplot(data=df_tel, x='data.status', y='data.temperature', palette=['#2ecc71', '#e74c3c'])
plt.title('Operating Temperature (°C) by Machine Health Status', fontsize=12, fontweight='bold')
plt.xlabel('Telemetry Status')
plt.ylabel('Temperature (°C)')
plt.show()


## 7. Portfolio Visualization Asset
Generate dual-panel bar charts displaying operational bottlenecks.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Factory Downtime
colors_fac = ['#2b5c8f' if 'seiko' in f else '#6baed6' for f in factory_summary['location.factory']]
axes[0].bar(factory_summary['location.factory'].str.replace('daikibo-', ''), factory_summary['total_downtime_min'], color=colors_fac, edgecolor='black')
axes[0].set_title('Total Unscheduled Downtime by Factory (Mins)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Downtime (min)')
for idx, val in enumerate(factory_summary['total_downtime_min']):
    axes[0].text(idx, val + 10, f"{val}m", ha='center', fontweight='bold')

# Plot 2: Device Downtime
colors_dev = ['#d73027' if v > 400 else '#fee08b' for v in device_summary['total_downtime_min']]
axes[1].bar(device_summary['deviceType'], device_summary['total_downtime_min'], color=colors_dev, edgecolor='black')
axes[1].set_title('Global Downtime by Device Type (Mins)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Downtime (min)')
axes[1].tick_params(axis='x', rotation=35)
for idx, val in enumerate(device_summary['total_downtime_min']):
    if val > 0:
        axes[1].text(idx, val + 10, f"{val}m", ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


## 8. Export Processed Data for Tableau
Save normalized data to `../data/processed/telemetry_clean.csv`.


In [ ]:
os.makedirs('../data/processed', exist_ok=True)
df_tel.to_csv('../data/processed/telemetry_clean.csv', index=False)
print("Export complete: ../data/processed/telemetry_clean.csv")
